In [ ]:
# =========================================================
# 2. 무신사 상품 메타데이터 크롤링 - 100개씩 이어서 저장하는 버전 v4
# =========================================================
# 사용법:
# - BATCH_SIZE = 100 으로 두면 실행할 때마다 "아직 수집 안 된 다음 100개"만 수집
# - 같은 셀을 다시 실행하면 기존 raw CSV를 읽고, 이미 수집된 product_id는 자동 스킵
# - 전체가 끝날 때까지 반복 실행하면 됨
#
# DB product 테이블용 CSV 컬럼:
# product_id, product_name, brand_id, category_id, original_price, sale_price,
# discount_rate, gender, rating, wish_count, review_count, cumulative_sales,
# img_url, name_emb, image_emb
# =========================================================

from pathlib import Path
import pandas as pd
import re
import time
import random
from datetime import datetime

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# ---------------------------------------------------------
# 0) 기본 설정
# ---------------------------------------------------------

BASE_DIR = Path(r"C:\Users\vixxb\Desktop\MUSINSA_WEB-main\musinsa-crawler-lite\crawlers")

# 여기만 바꿔가면서 실행
STYLE_TAG = "A_미니멀"

CSV_PATH = BASE_DIR / f"musinsa_goods_nos_{STYLE_TAG}_ordered.csv"

OUTPUT_DIR = BASE_DIR / "metadata_output"
OUTPUT_DIR.mkdir(exist_ok=True)

# 실행 1번당 수집할 개수
BATCH_SIZE = 100

# 빠르게 돌릴 때 True 권장
# 에러 원인을 눈으로 확인해야 하면 False
HEADLESS = True

# 몇 개마다 중간 저장할지
CHECKPOINT_EVERY = 20

# True면 기존 저장 파일을 읽고 이어서 수집
RESUME = True

# 대량 수집 시 xlsx는 느려서 비추천
SAVE_XLSX = False

# 이어서 저장하려면 파일명을 고정해야 함
# test/full 같은 suffix를 붙이면 이어받기가 꼬일 수 있음
PRODUCT_TABLE_CSV_PATH = OUTPUT_DIR / f"musinsa_product_table_{STYLE_TAG}.csv"
MAPPING_CSV_PATH = OUTPUT_DIR / f"musinsa_product_mapping_{STYLE_TAG}.csv"
RAW_CSV_PATH = OUTPUT_DIR / f"musinsa_product_crawl_raw_{STYLE_TAG}.csv"
RAW_XLSX_PATH = OUTPUT_DIR / f"musinsa_product_crawl_raw_{STYLE_TAG}.xlsx"


# ---------------------------------------------------------
# 1) CSV 불러오기
# ---------------------------------------------------------

df = pd.read_csv(CSV_PATH)

# goodsNo = product_id
df["product_id"] = df["goodsNo"].astype(str).str.strip()
df["style_tag"] = STYLE_TAG

# product_url이 CSV에 없거나 비어있으면 생성
if "product_url" not in df.columns:
    df["product_url"] = df["product_id"].apply(lambda x: f"https://www.musinsa.com/products/{x}")
else:
    df["product_url"] = df["product_url"].fillna("").astype(str)
    empty_url_mask = df["product_url"].str.strip() == ""
    df.loc[empty_url_mask, "product_url"] = df.loc[empty_url_mask, "product_id"].apply(
        lambda x: f"https://www.musinsa.com/products/{x}"
    )

# snap_url 없을 경우 대비
if "snap_url" not in df.columns:
    df["snap_url"] = ""
else:
    df["snap_url"] = df["snap_url"].fillna("").astype(str)

# 중복 상품 제거
df = df.drop_duplicates(subset=["product_id"]).reset_index(drop=True)


# ---------------------------------------------------------
# 2) 기존 결과 이어받기
# ---------------------------------------------------------

results = []
done_ids = set()

if RESUME and RAW_CSV_PATH.exists():
    prev_df = pd.read_csv(RAW_CSV_PATH, dtype={"product_id": str})
    prev_df["product_id"] = prev_df["product_id"].astype(str).str.strip()
    results = prev_df.to_dict("records")
    done_ids = set(prev_df["product_id"].tolist())
    print(f"기존 결과 로드 완료: {len(done_ids)}개")

# 아직 수집하지 않은 상품만 필터링
df_pending = df[~df["product_id"].isin(done_ids)].copy().reset_index(drop=True)

# 이번 실행에서 수집할 다음 100개
df_batch = df_pending.head(BATCH_SIZE).copy()

print("전체 상품 수:", len(df))
print("이미 수집한 상품 수:", len(done_ids))
print("남은 상품 수:", len(df_pending))
print("이번 실행 수집 수:", len(df_batch))

if len(df_batch) == 0:
    print("수집할 상품이 없습니다. 이미 완료된 것으로 보입니다.")

display(df_batch.head())


# ---------------------------------------------------------
# 3) 공통 유틸 함수
# ---------------------------------------------------------

def clean_text(value):
    if value is None:
        return None
    value = str(value).replace("\n", " ").replace("\t", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value if value else None


def clean_brand_name(value):
    text = clean_text(value)
    if text is None:
        return None
    text = re.sub(r"\s*(무신사\s*)?단독\s*$", "", text).strip()
    return text if text else None


def parse_number_with_unit(value):
    """
    '1.8천' -> 1800
    '34.6만' -> 346000
    '3,421' -> 3421
    '999+' -> 999
    """
    if value is None:
        return None

    text = str(value).strip()
    text = text.replace(",", "").replace("+", "")
    text = text.replace("원", "").replace("개", "")
    text = text.replace("찜", "").replace("후기", "").replace("리뷰", "").replace("스냅", "")
    text = text.strip()

    match_man = re.search(r"(\d+(?:\.\d+)?)\s*만", text)
    if match_man:
        return int(float(match_man.group(1)) * 10000)

    match_cheon = re.search(r"(\d+(?:\.\d+)?)\s*천", text)
    if match_cheon:
        return int(float(match_cheon.group(1)) * 1000)

    match_num = re.search(r"\d+(?:\.\d+)?", text)
    if match_num:
        return int(float(match_num.group(0)))

    return None


def parse_price(value):
    """
    가격 텍스트를 정수로 변환.
    0~100 사이 숫자는 할인율일 가능성이 높으므로 가격으로 인정하지 않음.
    """
    if value is None:
        return None

    text = str(value).strip().replace(",", "")
    price_candidates = []

    # 원 단위 숫자 우선
    won_matches = re.findall(r"(\d+(?:\.\d+)?)\s*원", text)
    for num in won_matches:
        n = int(float(num))
        if n > 100:
            price_candidates.append(n)

    if price_candidates:
        return price_candidates[-1]

    # 원 문자가 없는 경우 대비
    num_matches = re.findall(r"\d+(?:\.\d+)?", text)
    for num in num_matches:
        n = int(float(num))
        if n > 100:
            price_candidates.append(n)

    if price_candidates:
        return price_candidates[-1]

    return None


def parse_float(value):
    if value is None:
        return None
    text = str(value).strip()
    match = re.search(r"\d+(?:\.\d+)?", text)
    if match:
        return float(match.group(0))
    return None


def parse_discount_rate(value):
    if value is None:
        return 0
    text = str(value).strip()
    match = re.search(r"\d+", text)
    if match:
        rate = int(match.group(0))
        if 0 <= rate <= 100:
            return rate
    return 0


def parse_review_count(value):
    text = clean_text(value)
    if text is None:
        return 0

    # 스냅 n개는 후기 없음
    if "스냅" in text and "후기" not in text:
        return 0

    parsed = parse_number_with_unit(text)
    return parsed if parsed is not None else 0


def calc_discount_rate(original_price, sale_price):
    if original_price and sale_price and original_price > sale_price:
        return round((original_price - sale_price) / original_price * 100)
    return 0


def normalize_gender(value):
    """
    공용 -> 0
    남 / 남성 -> 1
    여 / 여성 -> 2
    """
    text = clean_text(value)
    if text is None:
        return None

    text = text.replace(" ", "")

    if "공용" in text:
        return 0

    # 남성과 여성이 동시에 있으면 공용
    if ("남성" in text and "여성" in text) or ("남" in text and "여" in text):
        return 0

    if text in ["여", "여성"] or "여성" in text:
        return 2

    if text in ["남", "남성"] or "남성" in text:
        return 1

    return None


def safe_find(driver, css_selector, timeout=3):
    try:
        return WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, css_selector))
        )
    except Exception:
        return None


def safe_text(driver, css_selector, timeout=2):
    elem = safe_find(driver, css_selector, timeout=timeout)
    if elem is None:
        return None
    return clean_text(elem.text)


def safe_text_any(driver, css_selectors, timeout=2):
    for selector in css_selectors:
        text = safe_text(driver, selector, timeout=timeout)
        if text:
            return text
    return None


def safe_attr(driver, css_selector, attr_name, timeout=2):
    elem = safe_find(driver, css_selector, timeout=timeout)
    if elem is None:
        return None
    value = elem.get_attribute(attr_name)
    return clean_text(value)


def safe_xpath_text(driver, xpath, timeout=2):
    try:
        elem = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((By.XPATH, xpath))
        )
        return clean_text(elem.text)
    except Exception:
        return None


def get_gender_text(driver):
    """
    성별 수집 전용.
    무신사 상세 정보 영역의 <dt>성별</dt><dd>여</dd> 구조 우선.
    """
    # 성별 영역이 아래쪽에 있으므로 살짝 스크롤
    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.45);")
        time.sleep(0.2)
    except Exception:
        pass

    # 1순위: <div><dt>성별</dt><dd>여</dd></div>
    gender_text = safe_xpath_text(
        driver,
        "//div[./dt[normalize-space()='성별']]/dd",
        timeout=2
    )
    if gender_text:
        return gender_text

    # 2순위: dt 성별 다음 dd
    gender_text = safe_xpath_text(
        driver,
        "//dt[normalize-space()='성별']/following-sibling::dd[1]",
        timeout=2
    )
    if gender_text:
        return gender_text

    # 3순위: dt에 성별 포함
    gender_text = safe_xpath_text(
        driver,
        "//dt[contains(normalize-space(), '성별')]/following-sibling::dd[1]",
        timeout=2
    )
    if gender_text:
        return gender_text

    # 4순위: fallback CSS
    gender_text = safe_text_any(
        driver,
        [
            "div[class^='VariableArea__Container'] dl div:nth-child(2) dd",
            "div[class^='ContentsLayout__Inner'] dl div:nth-child(2) dd",
            "dl div:nth-child(2) dd",
        ],
        timeout=1
    )

    return gender_text


# ---------------------------------------------------------
# 4) 저장 함수
# ---------------------------------------------------------

PRODUCT_COLUMNS = [
    "product_id",
    "product_name",
    "brand_id",
    "category_id",
    "original_price",
    "sale_price",
    "discount_rate",
    "gender",
    "rating",
    "wish_count",
    "review_count",
    "cumulative_sales",
    "img_url",
    "name_emb",
    "image_emb",
]

MAPPING_COLUMNS = [
    "product_id",
    "style_tag",
    "snap_url",
    "product_url",
    "brand_name",
    "brand_url",
    "upper_category",
    "lower_category",
    "gender_text",
    "gender",
    "crawl_status",
    "error_message",
    "crawled_at",
]


def build_output_dfs(results):
    raw_df = pd.DataFrame(results)

    if raw_df.empty:
        product_df = pd.DataFrame(columns=PRODUCT_COLUMNS)
        mapping_df = pd.DataFrame(columns=MAPPING_COLUMNS)
        return raw_df, product_df, mapping_df

    # 누락 컬럼 대비
    for col in PRODUCT_COLUMNS + MAPPING_COLUMNS:
        if col not in raw_df.columns:
            raw_df[col] = None

    raw_df["product_id"] = pd.to_numeric(raw_df["product_id"], errors="coerce").astype("Int64")

    product_df = raw_df[PRODUCT_COLUMNS].copy()
    mapping_df = raw_df[MAPPING_COLUMNS].copy()

    int_cols = [
        "product_id",
        "brand_id",
        "category_id",
        "original_price",
        "sale_price",
        "discount_rate",
        "gender",
        "wish_count",
        "review_count",
        "cumulative_sales",
    ]

    for col in int_cols:
        product_df[col] = pd.to_numeric(product_df[col], errors="coerce").astype("Int64")

    product_df["rating"] = pd.to_numeric(product_df["rating"], errors="coerce")

    return raw_df, product_df, mapping_df


def save_outputs(results):
    raw_df, product_df, mapping_df = build_output_dfs(results)

    product_df.to_csv(PRODUCT_TABLE_CSV_PATH, index=False, encoding="utf-8-sig")
    mapping_df.to_csv(MAPPING_CSV_PATH, index=False, encoding="utf-8-sig")
    raw_df.to_csv(RAW_CSV_PATH, index=False, encoding="utf-8-sig")

    if SAVE_XLSX:
        raw_df.to_excel(RAW_XLSX_PATH, index=False)

    return raw_df, product_df, mapping_df


# ---------------------------------------------------------
# 5) Selenium 드라이버 실행
# ---------------------------------------------------------

if len(df_batch) > 0:
    chrome_options = Options()

    if HEADLESS:
        chrome_options.add_argument("--headless=new")

    chrome_options.add_argument("--window-size=1400,1200")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--lang=ko-KR")
    chrome_options.add_argument("--disable-notifications")
    chrome_options.add_argument("--disable-popup-blocking")

    # 속도 개선. 이미지 파일 자체 다운로드를 줄임.
    # img_url은 대부분 DOM의 src에서 수집 가능.
    chrome_options.add_argument("--blink-settings=imagesEnabled=false")

    # 전체 로딩을 기다리지 않고 DOMContentLoaded 수준에서 진행
    chrome_options.page_load_strategy = "eager"

    chrome_options.add_argument(
        "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(options=chrome_options)
else:
    driver = None


# ---------------------------------------------------------
# 6) 상품 1개 크롤링 함수
# ---------------------------------------------------------

def crawl_musinsa_product(driver, row):
    product_id = str(row["product_id"]).strip()
    product_url = row["product_url"]
    snap_url = row.get("snap_url", "")

    result = {
        # 원본 추적용
        "product_id": product_id,
        "style_tag": STYLE_TAG,
        "snap_url": snap_url,
        "product_url": product_url,

        # 매핑용
        "brand_name": None,
        "brand_url": None,
        "upper_category": None,
        "lower_category": None,
        "gender_text": None,

        # DB product 테이블용
        "product_name": None,
        "brand_id": None,
        "category_id": None,
        "original_price": None,
        "sale_price": None,
        "discount_rate": 0,
        "gender": None,
        "rating": None,
        "wish_count": 0,
        "review_count": 0,
        "cumulative_sales": 0,
        "img_url": None,
        "name_emb": None,
        "image_emb": None,

        # 로그
        "crawl_status": "failed",
        "error_message": None,
        "crawled_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    try:
        driver.get(product_url)

        WebDriverWait(driver, 8).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "body"))
        )
        time.sleep(random.uniform(0.5, 0.9))

        # 상품명
        result["product_name"] = safe_text(
            driver,
            "div[class^='GoodsName__Wrap'] span",
            timeout=4
        )

        # 브랜드명 / 브랜드 URL
        brand_text = safe_text_any(
            driver,
            [
                "div[class^='Brand__Wrap'] a span span.font-global",
                "div[class^='Brand__Wrap'] a span span[class*='font-global']",
                "div[class^='Brand__Wrap'] a span span",
                "div[class^='Brand__Wrap'] a",
            ],
            timeout=2
        )
        result["brand_name"] = clean_brand_name(brand_text)

        result["brand_url"] = safe_attr(
            driver,
            "div[class^='Brand__Wrap'] a",
            "href",
            timeout=2
        )

        # 카테고리
        result["upper_category"] = safe_text(
            driver,
            "div[class^='Category__Wrap'] span:nth-child(1) a",
            timeout=2
        )

        result["lower_category"] = safe_text(
            driver,
            "div[class^='Category__Wrap'] span:nth-child(2) a",
            timeout=2
        )

        # 성별
        gender_text = get_gender_text(driver)
        result["gender_text"] = gender_text
        result["gender"] = normalize_gender(gender_text)

        # 가격
        original_price_text = safe_text_any(
            driver,
            [
                "div[class^='Price__DiscountWrap'] span[class*='line-through']",
                "span[class*='line-through'][class*='text-gray-500']",
                "div[class^='Price__DiscountWrap'] span",
            ],
            timeout=1
        )

        sale_price_text = safe_text_any(
            driver,
            [
                "span[class*='Price__CalculatedPrice']",
                "div[class^='Price__CurrentPrice'] span[class*='Price__CalculatedPrice']",
                "div[class^='Price__CurrentPrice'] span[class*='text-black']",
                "div[class^='Price__CurrentPrice']",
            ],
            timeout=2
        )

        discount_rate_text = safe_text_any(
            driver,
            [
                "span[class*='Price__DiscountRate']",
                "div[class^='Price__CurrentPrice'] span[class*='text-red']",
            ],
            timeout=1
        )

        original_price = parse_price(original_price_text)
        sale_price = parse_price(sale_price_text)
        discount_rate = parse_discount_rate(discount_rate_text)

        if sale_price is None:
            current_price_full_text = safe_text(driver, "div[class^='Price__CurrentPrice']", timeout=2)
            sale_price = parse_price(current_price_full_text)

        if original_price is None and sale_price is not None:
            original_price = sale_price

        if original_price is not None and original_price <= 100:
            original_price = sale_price

        if sale_price is not None and sale_price <= 100:
            sale_price = None

        if discount_rate == 0:
            discount_rate = calc_discount_rate(original_price, sale_price)

        result["original_price"] = original_price
        result["sale_price"] = sale_price
        result["discount_rate"] = discount_rate

        # 대표 이미지 URL
        result["img_url"] = safe_attr(
            driver,
            "div.swiper-slide-active img",
            "src",
            timeout=2
        )

        if result["img_url"] is None:
            result["img_url"] = safe_attr(
                driver,
                "img[src*='goods_img']",
                "src",
                timeout=2
            )

        # 평점 / 리뷰 수
        rating_text = safe_text(
            driver,
            "div[class^='ReviewSummary__TotalWrap'] span[class*='font-[500]']",
            timeout=1
        )

        review_count_text = safe_text_any(
            driver,
            [
                "div[class^='ReviewSummary__TotalWrap'] div[class^='ReviewSummary__Wrap'] span[class*='underline']",
                "div[class^='ReviewSummary__TotalWrap'] span[class*='underline']",
            ],
            timeout=1
        )

        result["rating"] = parse_float(rating_text)
        result["review_count"] = parse_review_count(review_count_text)

        # 찜 수
        wish_text = safe_text(
            driver,
            "div[class^='Purchase__Container'] span",
            timeout=1
        )
        parsed_wish = parse_number_with_unit(wish_text)
        result["wish_count"] = parsed_wish if parsed_wish is not None else 0

        # 누적 판매량: selector 미확인. 기존 DB 형식에 맞춰 0
        result["cumulative_sales"] = 0

        required = ["product_id", "product_name", "original_price", "sale_price", "img_url"]
        missing_required = [col for col in required if result.get(col) in [None, ""]]

        if missing_required:
            result["crawl_status"] = "partial"
            result["error_message"] = f"필수값 일부 누락: {missing_required}"
        else:
            result["crawl_status"] = "success"

    except Exception as e:
        result["crawl_status"] = "failed"
        result["error_message"] = str(e)[:500]

    return result


# ---------------------------------------------------------
# 7) 이번 배치 크롤링 실행
# ---------------------------------------------------------

try:
    if len(df_batch) > 0:
        for i, (_, row) in enumerate(df_batch.iterrows(), start=1):
            pid = str(row["product_id"]).strip()

            print(f"[{i}/{len(df_batch)}] 크롤링 중: {pid}")

            item = crawl_musinsa_product(driver, row)
            results.append(item)
            done_ids.add(pid)

            print(
                "  →",
                item["crawl_status"],
                "|",
                item["product_name"],
                "| price:",
                item["sale_price"],
                "| gender:",
                item["gender_text"],
                "->",
                item["gender"],
                "| brand:",
                item["brand_name"]
            )

            if len(results) % CHECKPOINT_EVERY == 0:
                save_outputs(results)
                print("  중간 저장 완료:", len(results), "개")

            time.sleep(random.uniform(0.25, 0.6))

finally:
    if driver is not None:
        driver.quit()


# ---------------------------------------------------------
# 8) 최종 저장 및 확인
# ---------------------------------------------------------

raw_df, product_df, mapping_df = save_outputs(results)

print("\n저장 완료")
print("product 테이블용 CSV:", PRODUCT_TABLE_CSV_PATH)
print("브랜드/카테고리 매핑용 CSV:", MAPPING_CSV_PATH)
print("전체 원본 CSV:", RAW_CSV_PATH)

print("\n현재까지 누적 수집 수:", len(raw_df))
print("이번 실행 후 남은 예상 수:", max(len(df) - len(raw_df), 0))

print("\nproduct 테이블용 결과 미리보기")
display(product_df.tail())

print("\n매핑/검수용 결과 미리보기")
display(mapping_df.tail())

print("\n수집 상태 요약")
display(raw_df["crawl_status"].value_counts(dropna=False).to_frame("count"))

print("\ngender 수집 요약")
if "gender_text" in raw_df.columns:
    display(raw_df[["gender_text", "gender"]].value_counts(dropna=False).to_frame("count"))
